In [ ]:
# cell 1
import pandas as pd

In [13]:
# cell 2
df = pd.read_parquet("data/dataset.parquet")

In [16]:
import pandas as pd

# Extract numbers from all_text, explode into a series, exclude '00'
text_nums = df['all_text'].str.findall(r'\b\d+\b').explode()
text_nums = text_nums[text_nums != '00']

# Extract timestamp components (no year), zero-pad, stack into a series, exclude '00'
ts_nums = pd.DataFrame({
    'month':  df['timestamp'].dt.month.astype(str).str.zfill(2),
    'day':    df['timestamp'].dt.day.astype(str).str.zfill(2),
    'hour':   df['timestamp'].dt.hour.astype(str).str.zfill(2),
    'minute': df['timestamp'].dt.minute.astype(str).str.zfill(2),
    'second': df['timestamp'].dt.second.astype(str).str.zfill(2),
}).stack()
ts_nums = ts_nums[ts_nums != '00']

# Combine, count, take top 50
top50 = (
    pd.concat([text_nums, ts_nums])
    .value_counts()
    .head(50)
    .rename_axis('Number')
    .reset_index(name='Count')
)
top50.index += 1
top50.index.name = 'Rank'
print(top50.to_string())


     Number  Count
Rank              
1         1   7694
2        01   6850
3        10   6812
4        12   6444
5        02   6438
6        03   6294
7        11   5972
8        05   5837
9        04   5590
10       08   5414
11       20   5027
12       09   4880
13       06   4559
14       07   4537
15       19   4392
16       17   4355
17       22   4354
18     2024   4347
19       15   4287
20       18   4285
21       23   4285
22       16   4209
23       21   3980
24        2   3911
25       14   3618
26       13   3363
27        3   2980
28        4   2655
29       30   2540
30       24   2384
31       25   2315
32       26   2151
33        5   2069
34       29   2008
35       28   2005
36     2023   1975
37        6   1967
38       27   1852
39      000   1798
40       31   1682
41        7   1460
42       50   1326
43        8   1308
44     2020   1270
45        9   1143
46       40   1028
47     2025    978
48     2022    970
49       47    939
50       45    902


In [17]:
def count_num(series, n):
    return series.fillna('').str.count(rf'\b{n}\b').sum()

text_count  = count_num(df['text'], 22)
image_count = count_num(df['image_descriptions'], 22)
video_count = count_num(df['video_transcripts'], 22)

ts_count = (
    (df['timestamp'].dt.month  == 22).sum() +
    (df['timestamp'].dt.day    == 22).sum() +
    (df['timestamp'].dt.hour   == 22).sum() +
    (df['timestamp'].dt.minute == 22).sum() +
    (df['timestamp'].dt.second == 22).sum()
)

total = text_count + image_count + video_count + ts_count
print(f'Post text:          {text_count:>6,}')
print(f'Image descriptions: {image_count:>6,}')
print(f'Video transcripts:  {video_count:>6,}')
print(f'Timestamps:         {ts_count:>6,}')
print(f'─────────────────────────')
print(f'Total:              {total:>6,}')


Post text:             169
Image descriptions:    325
Video transcripts:      61
Timestamps:          3,799
─────────────────────────
Total:               4,354
